# Python 101
## Part XI.
---

## More pandas are coming!

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
BASE_URI = "./data/"

---
### Act I: Read demographic data

The data comes from the World Bank: the total population of every country, 1960-2025
(indicator `SP.POP.TOTL`). Both data files of this chapter were downloaded with
`python101_11_data_scraping.py` - open it to see how.

- Read the data from a csv file

In [ ]:
df = pd.read_csv(BASE_URI + 'population.csv')

In [ ]:
df

- Select the columns we need: the country name and the years 1960-2025
- `.dropna()` removes every row that has a missing value

In [ ]:
columns = ['Country Name'] + [str(year) for year in range(1960, 2026)]
pop = df[columns].dropna()
pop.head()

- Rename `'Country Name'` to `'country'` and set it as index.

In [ ]:
pop = pop.rename(columns={'Country Name': 'country'}).set_index('country')
pop.head()

- Select a few countries: `isin(countries)` is `True` for the rows whose country is in the list

In [ ]:
countries = ['United Kingdom', 'Hungary', 'France', 'Germany']
subpop = pop.loc[pop.index.isin(countries)]
subpop

- Transpose the dataframe: rows become columns and columns become rows,
  so we get one row per year and one column per country - the shape `.plot()` wants

In [ ]:
subpop = subpop.transpose()
subpop.head()

- Plot!

In [ ]:
subpop.plot.line();

---
### Act II: Read data about alcohol

The data comes from the World Health Organization (WHO): the recorded alcohol consumption per
person aged 15 or over, in litres of pure alcohol, by type of drink, for the countries of
the WHO Europe region.

- Read the data

In [ ]:
df = pd.read_csv(BASE_URI + 'alcohol.csv')
df.head()

- Select the columns, and rename them

In [ ]:
columns = {
    'Country': 'country',
    'Year': 'year',
    'Beverage Types': 'type',
    'Display Value': 'alcohol'
}
alc = df[list(columns.keys())].rename(columns=columns).dropna()
alc.head()

- Select the same country subset

In [ ]:
subalc = alc.loc[alc['country'].isin(countries)]
subalc.head()

- Check which countries we got:

In [ ]:
subalc['country'].unique()

Only three! The United Kingdom is missing: the WHO calls it *United Kingdom of Great Britain
and Northern Ireland*, the World Bank just *United Kingdom*. Two sources, two names for the
same country - a common trap when you combine data.

- Rename it in `alc` with `replace`, as we shortened the party names in chapter X,
  and select the countries again:

In [ ]:
alc['country'] = alc['country'].replace(
    {'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom'})

subalc = alc.loc[alc['country'].isin(countries)]
subalc['country'].unique()

We only care about the combined consumption:
- filter the dataframe, select rows where `type` is `'All types'`
- remove the `type` column: it only holds `'All types'` now

In [ ]:
subalc = subalc.loc[subalc['type'] == 'All types'].drop(columns=['type'])
subalc.head()

- Pivot the dataframe, as in chapter X: one row per year, one column per country

In [ ]:
subalc = subalc.pivot(index='year', columns='country', values='alcohol')
subalc.head()

- Plot!

In [ ]:
subalc.plot();

---
### Act III: Merge data

We will join the two dataframes on their index, the years. That only works if the years
are the same type of value in both.

- Check the index types

In [ ]:
print('subpop index type:', subpop.index.dtype)
print('subalc index type:', subalc.index.dtype)

`subpop`'s index holds the years as text (`object`), `subalc`'s as integers (`int64`).
Change `subpop`'s to integers:

In [ ]:
subpop.index = subpop.index.astype('int')

- Check which years only one of the two dataframes has (the *symmetric difference*)

In [ ]:
subpop.index.symmetric_difference(subalc.index)

The population data runs from 1960 to 2025, the alcohol data of our four countries from 1961 to 2024.

- Keep the years both have: 1961 to 2024.
  `&` means *and*: a row is kept only if both conditions are true

In [ ]:
subpop = subpop.loc[(1960 < subpop.index) & (subpop.index < 2025)]
subalc = subalc.loc[(1960 < subalc.index) & (subalc.index < 2025)]

- Join the two dataframes! Both have a column for every country, so `join` needs a suffix for
  the alcohol columns: `rsuffix='_alc'` turns the second `Hungary` into `Hungary_alc`

In [ ]:
merged = subpop.join(subalc, rsuffix='_alc')
merged.head()

- Plot the two Hungarian columns into separate charts with `subplots=True`

In [ ]:
merged[['Hungary', 'Hungary_alc']].plot.line(subplots=True);

- Estimate the total alcohol consumption of each country: population times litres per person.
  It is only a rough estimate: the alcohol figures are litres of pure alcohol per person
  aged 15 or over, and we multiply them by the whole population, children included.

In [ ]:
for country in countries:
    merged[country + '_consumption'] = merged[country] * merged[country + '_alc']

- and plot it!

In [ ]:
merged[[c + '_consumption' for c in countries]].plot();

---

## Let's do some...

<img align="left" width=150 src="pics/magic.gif">
<br style="clear:left;"/>

### Act IV: Cool library of the week: <a href="https://github.com/Data-Centric-AI-Community/fg-data-profiling">fg-data-profiling</a>
#### Generate detailed reports from pandas dataframes
- import it
- generate a report

Install with:
```bash
pip install fg-data-profiling
```

> *Already installed* - `requirements.txt` covers this. You only need the command
> above on a new machine or in a fresh environment. If you see `No module named ...`,
> you are on the wrong kernel: **Select Kernel** -> **python101**.

It used to be called `ydata-profiling` (and before that `pandas-profiling`), so older
tutorials write `import ydata_profiling` - here it is `import data_profiling`.
If you see `No module named 'data_profiling'` on the right kernel, your environment was
installed with the old `requirements.txt`. Swap the two packages once, in the
Anaconda Prompt (Windows) or the Terminal (macOS):
```bash
conda activate python101
pip uninstall -y ydata-profiling
pip install fg-data-profiling
```

In [ ]:
import data_profiling

In [ ]:
profile = data_profiling.ProfileReport(alc)
profile

Export it to a file, and open that file in your browser:

In [ ]:
profile.to_file('./alc_profile.html')

---
## Final Act: your play time!
## It's your turn - write the missing code snippets!

#### 1. Plot the top 5 alcohol consuming countries in 1990, with their consumption

Solution steps:
1. Start from `alc`: it has every country, not just our four
2. Keep the rows from 1990 where `type` is `'All types'`. These are two conditions at once: join them
   with `&` and put each in its own brackets, as in Act III
3. Sort by `alcohol`, descending, and keep the first 5 rows
4. Set `country` as the index (as in Act I), so the bars get the country names, and plot the
   `alcohol` column as a bar chart: `.plot(kind='bar')`

#### 2. Compare the average alcohol consumption of France, Germany, the United Kingdom and Hungary on a plot

Solution steps:
1. Start from `subalc`: it already has one column per country, for the years 1961 to 2024
2. `.mean()` works like `.sum()`, but gives the average. On a dataframe it gives one average
   per column - here, one per country
3. Sort the averages, descending. They are a single column, so there is no column name to give:
   `.sort_values(ascending=False)`
4. Plot them as a bar chart. Which country drank the most?

#### 3. Which country consumed the most spirits in each year?

Solution steps:
1. Keep the rows of `alc` where `type` is `'Spirits'`
2. Sort them by `alcohol`, descending
3. Group by `year` and keep the first row of every group with `.first()` - after the sort,
   that is the country with the most spirits (as in exercise 7 of chapter X)
4. Which country came first most often? Use `value_counts()` on the `country` column

#### 4. How did the consumption of the different types of alcohol change over time in Hungary? Plot them!

Solution steps:
1. Keep the rows of `alc` where `country` is `'Hungary'`
2. Pivot them, as in Act II: one row per `year`, one column per `type`, with the `alcohol` values
3. Plot the result
4. Which type of drink fell the most between 1990 and 2024?

#### 5. Whose combined consumption changed the most between 1990 and 2024? (top 5)

Solution steps:
1. Keep the rows of `alc` where `type` is `'All types'`
2. Pivot them: one row per `country`, one column per `year`, with the `alcohol` values
3. Subtract the 1990 column from the 2024 column, and store the result in a new column called
   `change`. The year columns are numbers here, so it is `[2024]`, not `['2024']`
4. Some countries went up, some went down - and both count. `np.abs()` turns every number
   positive (`np.abs(-7.5)` is `7.5`), so store `np.abs()` of `change` in another new column,
   `abs_change`
5. Sort by `abs_change`, descending, and look at the top 5. Which of them went up, and which went down?